In [2]:
!pip install streamlit
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 101.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.3 MB/s eta 0:00:00


In [3]:
## Mounting and connecting to dataset
from google.colab import drive
print("Mounting Google Drive connection...")
drive.mount('/content/drive')

Mounting Google Drive connection...
Mounted at /content/drive


In [7]:
"""
╔══════════════════════════════════════════════════════════════════════════╗
║   Agri-Vision: Fall Armyworm Detection & Analytics Dashboard           ║
║   ─────────────────────────────────────────────────────────────────    ║
║   Inputs  : Image | Video | Webcam                                     ║
║   Model   : YOLOv8  (best.pt from training pipeline)                   ║
║   Classes : 0 healthy_maize | 1 early_stage | 2 late_stage            ║
║   Outputs : Live annotated feed · Per-class % stats ·                  ║
║             Agronomic recommendations · CSV + JSON export              ║
╚══════════════════════════════════════════════════════════════════════════╝
"""

# ── stdlib ────────────────────────────────────────────────────────────────────
import collections
import io
import json
import os
import tempfile
import time

# ── third-party ───────────────────────────────────────────────────────────────
import cv2
import numpy as np
import pandas as pd
import streamlit as st
from PIL import Image


!npm install -g localtunnel

# 2. Write your app to a file
%%writefile app.py

# ─────────────────────────────────────────────────────────────────────────────
# 0.  PAGE CONFIG  (must be the very first Streamlit call)
# ─────────────────────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Agri-Vision · FAW Detection",
    page_icon="🌽",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ─────────────────────────────────────────────────────────────────────────────
# 1.  GLOBAL STYLING
# ─────────────────────────────────────────────────────────────────────────────
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;600&family=DM+Sans:wght@300;400;600;800&display=swap');

/* ── root tokens ── */
:root{
  --bg:#0b0f14; --surface:#111820; --surface2:#192030;
  --border:#1e2d3d; --text:#dde6f0; --muted:#5a7080;
  --green:#00d17a; --amber:#f0a500; --red:#e03c3c;
  --blue:#3b8beb; --purple:#9b6dff;
}
html,body,[class*="css"]{font-family:'DM Sans',sans-serif;background:var(--bg);color:var(--text);}
#MainMenu,footer,header{visibility:hidden;}

/* header */
.av-header{
  background:linear-gradient(120deg,#061a0f 0%,#0b0f14 55%);
  border:1px solid var(--border); border-radius:14px;
  padding:28px 36px; margin-bottom:22px; position:relative; overflow:hidden;
}
.av-header::after{
  content:''; position:absolute; right:-60px; top:-60px;
  width:260px; height:260px; border-radius:50%;
  background:radial-gradient(circle,rgba(0,209,122,.13) 0%,transparent 70%);
}
.av-header h1{
  font-size:2rem; font-weight:800; margin:0 0 6px;
  background:linear-gradient(90deg,#00d17a,#00ffaa);
  -webkit-background-clip:text; -webkit-text-fill-color:transparent;
}
.av-header p{color:var(--muted);font-size:.9rem;margin:0;font-weight:300;}

/* metric pill */
.pill-row{display:flex;gap:10px;flex-wrap:wrap;margin:14px 0;}
.pill{
  flex:1; min-width:100px;
  background:var(--surface2); border:1px solid var(--border);
  border-radius:10px; padding:14px 18px; text-align:center;
}
.pill-val{font-family:'IBM Plex Mono',monospace;font-size:1.8rem;font-weight:600;line-height:1;}
.pill-lbl{font-size:.7rem;color:var(--muted);text-transform:uppercase;letter-spacing:.07em;margin-top:4px;}
.c-green{color:var(--green);} .c-amber{color:var(--amber);}
.c-red{color:var(--red);}     .c-blue{color:var(--blue);}

/* bar chart */
.bar-wrap{margin:6px 0;}
.bar-head{display:flex;justify-content:space-between;font-size:.8rem;margin-bottom:3px;}
.bar-track{height:11px;background:var(--border);border-radius:99px;overflow:hidden;}
.bar-fill{height:100%;border-radius:99px;}

/* recommendation card */
.reco{border-radius:12px;padding:20px 24px;margin-top:16px;border-left:5px solid;}
.reco h4{margin:0 0 10px;font-size:1rem;}
.reco p,.reco li{font-size:.85rem;color:var(--muted);line-height:1.75;margin:0;}
.reco ul{padding-left:18px;margin:8px 0 0;}
.r-green{background:rgba(0,209,122,.07);border-color:var(--green);}
.r-amber{background:rgba(240,165,0,.07);border-color:var(--amber);}
.r-red  {background:rgba(224,60,60,.07);border-color:var(--red);}
.r-blue {background:rgba(59,139,235,.07);border-color:var(--blue);}

/* sidebar */
section[data-testid="stSidebar"]{background:var(--surface);border-right:1px solid var(--border);}
/* buttons */
.stButton>button{
  background:var(--green)!important; color:#000!important;
  border:none!important; border-radius:8px!important;
  font-weight:700!important; width:100%!important; padding:10px!important;
}
/* tabs */
.stTabs [data-baseweb="tab-list"]{
  background:var(--surface2); border-radius:10px; padding:4px;
  border:1px solid var(--border);
}
.stTabs [data-baseweb="tab"]{border-radius:7px;color:var(--muted);font-weight:600;}
.stTabs [aria-selected="true"]{background:var(--green)!important;color:#000!important;}
hr{border-color:var(--border);}
</style>
""", unsafe_allow_html=True)

# ─────────────────────────────────────────────────────────────────────────────
# 2.  CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
CLASS_KEYS    = ["healthy_maize", "early_stage", "late_stage"]
CLASS_DISPLAY = {
    "healthy_maize": "Healthy Maize",
    "early_stage":   "Early Stage",
    "late_stage":    "Late Stage",
}
# BGR for OpenCV drawing
CLASS_BGR = {
    "healthy_maize": (0, 209, 122),
    "early_stage":   (0, 165, 240),
    "late_stage":    (60,  60, 224),
}
# HEX for HTML
CLASS_HEX = {
    "healthy_maize": "#00d17a",
    "early_stage":   "#f0a500",
    "late_stage":    "#e03c3c",
}
CLASS_ICON = {
    "healthy_maize": "🟢",
    "early_stage":   "🟡",
    "late_stage":    "🔴",
}
# Recommendation CSS class
CLASS_RCSS = {
    "healthy_maize": "r-green",
    "early_stage":   "r-amber",
    "late_stage":    "r-red",
}

# ─────────────────────────────────────────────────────────────────────────────
# 3.  SESSION STATE
# ─────────────────────────────────────────────────────────────────────────────
def _init_state():
    defaults = {
        "tracking_log":    [],          # list[dict] — every detection event
        "class_counts":    {k: 0 for k in CLASS_KEYS},
        "frame_count":     0,           # frames processed
        "session_start":   None,        # ISO timestamp when session began
        "camera_running":  False,
    }
    for key, val in defaults.items():
        if key not in st.session_state:
            st.session_state[key] = val

_init_state()

# ─────────────────────────────────────────────────────────────────────────────
# 4.  MODEL LOADING
# ─────────────────────────────────────────────────────────────────────────────
@st.cache_resource(show_spinner=False)
def load_yolo(path: str):
    try:
        from ultralytics import YOLO
        m = YOLO(path)
        return m, None
    except Exception as exc:
        return None, str(exc)

# ─────────────────────────────────────────────────────────────────────────────
# 5.  INFERENCE
# ─────────────────────────────────────────────────────────────────────────────
def run_inference(frame_rgb: np.ndarray,
                  model,
                  conf_thr: float,
                  iou_thr:  float) -> list[dict]:
    """
    Run detection on a single RGB frame.
    Returns list of {"key", "conf", "box":[x1,y1,x2,y2]}.
    Falls back to deterministic mock when model is None.
    """
    if model is not None:
        results = model.predict(
            source=frame_rgb, conf=conf_thr, iou=iou_thr,
            verbose=False, device="cpu",
        )
        out = []
        for box in (results[0].boxes or []):
            cid = int(box.cls)
            key = CLASS_KEYS[cid] if cid < len(CLASS_KEYS) else CLASS_KEYS[0]
            out.append({
                "key":  key,
                "conf": float(box.conf),
                "box":  box.xyxy[0].cpu().numpy().astype(int).tolist(),
            })
        return out

    # ── mock fallback ────────────────────────────────────────────────────
    h, w = frame_rgb.shape[:2]
    out  = []
    for _ in range(np.random.randint(1, 5)):
        cf = float(np.random.uniform(0.25, 0.97))
        if cf < conf_thr:
            continue
        key = np.random.choice(CLASS_KEYS, p=[0.30, 0.35, 0.35])
        x1  = np.random.randint(0, max(1, int(w * 0.45)))
        y1  = np.random.randint(0, max(1, int(h * 0.45)))
        x2  = min(np.random.randint(x1 + 30, x1 + int(w * 0.5) + 30), w - 1)
        y2  = min(np.random.randint(y1 + 30, y1 + int(h * 0.5) + 30), h - 1)
        out.append({"key": key, "conf": cf, "box": [x1, y1, x2, y2]})
    return out

# ─────────────────────────────────────────────────────────────────────────────
# 6.  DRAWING
# ─────────────────────────────────────────────────────────────────────────────
def draw_boxes(frame_bgr: np.ndarray, dets: list[dict]) -> np.ndarray:
    """
    Draw bounding boxes with confidence-scaled thickness & label on a BGR copy.
    Confidence level drives the box thickness (low=1 med=2 high=3).
    """
    out = frame_bgr.copy()
    for d in dets:
        key        = d["key"]
        cf         = d["conf"]
        x1,y1,x2,y2 = [int(v) for v in d["box"]]
        color      = CLASS_BGR.get(key, (200, 200, 200))
        thickness  = 1 if cf < 0.50 else 2 if cf < 0.75 else 3

        cv2.rectangle(out, (x1, y1), (x2, y2), color, thickness)

        label = f"{CLASS_DISPLAY[key]}  {cf:.0%}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.52, 2)
        by = max(y1 - 6, th + 6)
        cv2.rectangle(out, (x1, by - th - 4), (x1 + tw + 5, by + 2), color, -1)
        cv2.putText(out, label, (x1 + 3, by - 1),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.52, (0, 0, 0), 2, cv2.LINE_AA)
    return out

# ─────────────────────────────────────────────────────────────────────────────
# 7.  STATE UPDATERS
# ─────────────────────────────────────────────────────────────────────────────
def accumulate(dets: list[dict], frame_no: int, source_label: str):
    """Push detections into session state counters and log."""
    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    for d in dets:
        key = d["key"]
        st.session_state.class_counts[key] += 1
        st.session_state.tracking_log.append({
            "timestamp":    ts,
            "source":       source_label,
            "frame":        frame_no,
            "class_key":    key,
            "class_label":  CLASS_DISPLAY[key],
            "confidence":   round(d["conf"], 4),
            "box_x1":       d["box"][0],
            "box_y1":       d["box"][1],
            "box_x2":       d["box"][2],
            "box_y2":       d["box"][3],
        })
    st.session_state.frame_count += 1


def reset_session():
    st.session_state.tracking_log   = []
    st.session_state.class_counts   = {k: 0 for k in CLASS_KEYS}
    st.session_state.frame_count    = 0
    st.session_state.session_start  = None
    st.session_state.camera_running = False

# ─────────────────────────────────────────────────────────────────────────────
# 8.  ANALYTICS & RECOMMENDATION RENDERING
# ─────────────────────────────────────────────────────────────────────────────
def _bar_html(key: str, pct: float, count: int) -> str:
    color = CLASS_HEX[key]
    name  = CLASS_DISPLAY[key]
    icon  = CLASS_ICON[key]
    return (
        f'<div class="bar-wrap">'
        f'<div class="bar-head">'
        f'<span>{icon} {name}</span>'
        f'<span style="font-family:IBM Plex Mono,monospace;color:{color}">'
        f'{pct:.1f}%&nbsp;&nbsp;({count})</span></div>'
        f'<div class="bar-track">'
        f'<div class="bar-fill" style="width:{pct:.1f}%;background:{color}"></div>'
        f'</div></div>'
    )


def _reco_card(key: str, pct: float, extra: str = "") -> str:
    """Return HTML for a single recommendation card."""
    css   = CLASS_RCSS[key]
    icon  = CLASS_ICON[key]
    name  = CLASS_DISPLAY[key]

    bodies = {
        "healthy_maize": (
            "Field segment is within safe tolerance limits.",
            [
                "Continue weekly scouting during V1–V6 growth stages",
                "Maintain natural predator populations (earwigs, ants, spiders)",
                "Consider preventative <em>Bacillus thuringiensis</em> (Bt) application "
                "if neighbouring fields report FAW pressure",
                "Record scouting dates and GPS coordinates for traceability",
            ]
        ),
        "early_stage": (
            "Egg masses or early-instar larvae (L1–L3) detected. "
            "<strong>This is the optimal control window.</strong>",
            [
                "Apply <em>Bacillus thuringiensis</em> (Bt) or spinosad into the leaf whorl",
                "Chemical option: emamectin benzoate or chlorantraniliprole at label rate",
                "Scout full field — economic threshold: >20% plants infested",
                "Revisit in 5–7 days to confirm efficacy",
                "Early intervention can prevent up to 40% yield loss",
            ]
        ),
        "late_stage": (
            "Frass deposits and/or significant feeding damage detected. "
            "Mature larvae (L4–L6) are deeply embedded — <strong>act urgently</strong>.",
            [
                "Apply systemic insecticide (chlorpyrifos, lambda-cyhalothrin, or "
                "emamectin benzoate) with surfactant directly into the leaf whorl",
                "Do NOT spray leaf surfaces — target the whorl for penetration",
                "Assess damage level: if &gt;50% plants severely damaged at early "
                "growth stage, evaluate replanting",
                "Deploy pheromone traps to monitor adult moth population post-treatment",
                "Document damage zones for insurance and government support schemes",
            ]
        ),
    }

    title_body, bullets = bodies[key]
    bullet_html = "".join(f"<li>{b}</li>" for b in bullets)
    pct_badge   = (
        f'<span style="font-size:.78rem;font-weight:400;opacity:.65">'
        f'({pct:.1f}% of detections)</span>'
    )

    return (
        f'<div class="reco {css}">'
        f'<h4>{icon} {name} &nbsp;{pct_badge}</h4>'
        f'<p>{title_body}</p>'
        f'<ul>{bullet_html}</ul>'
        f'{extra}'
        f'</div>'
    )


def render_analytics(panel: st.delta_generator.DeltaGenerator,
                     counts: dict,
                     total:  int,
                     frames: int):
    """Render the full analytics + recommendation block into `panel`."""
    with panel.container():
        if total == 0:
            st.info("No detections yet.")
            return

        pct = {k: counts[k] / total * 100 for k in CLASS_KEYS}

        # ── pill metrics ─────────────────────────────────────────────────
        st.markdown(
            f'<div class="pill-row">'
            f'<div class="pill"><div class="pill-val c-blue">{total}</div>'
            f'<div class="pill-lbl">Total detections</div></div>'
            f'<div class="pill"><div class="pill-val c-blue">{frames}</div>'
            f'<div class="pill-lbl">Frames processed</div></div>'
            f'<div class="pill"><div class="pill-val c-green">{counts["healthy_maize"]}</div>'
            f'<div class="pill-lbl">Healthy</div></div>'
            f'<div class="pill"><div class="pill-val c-amber">{counts["early_stage"]}</div>'
            f'<div class="pill-lbl">Early stage</div></div>'
            f'<div class="pill"><div class="pill-val c-red">{counts["late_stage"]}</div>'
            f'<div class="pill-lbl">Late stage</div></div>'
            f'</div>',
            unsafe_allow_html=True,
        )

        # ── percentage bars ──────────────────────────────────────────────
        st.markdown("**Detection share (cumulative)**")
        bars_html = "".join(
            _bar_html(k, pct[k], counts[k])
            for k in CLASS_KEYS if counts[k] > 0
        )
        st.markdown(bars_html, unsafe_allow_html=True)

        st.markdown("---")

        # ── recommendation logic ─────────────────────────────────────────
        st.markdown("**💡 Agronomic Recommendation**")

        active       = {k: v for k, v in counts.items() if v > 0}
        active_pct   = {k: pct[k] for k in active}
        n_active     = len(active)

        # only one class detected
        if n_active == 1:
            dom = list(active.keys())[0]
            st.markdown(_reco_card(dom, pct[dom]), unsafe_allow_html=True)
            return

        # find dominant (highest count)
        max_count  = max(active.values())
        leaders    = [k for k, v in active.items() if v == max_count]

        # ── tie between two or more classes ─────────────────────────────
        if len(leaders) > 1:
            names = " & ".join(CLASS_DISPLAY[k] for k in leaders)
            tie_pct = pct[leaders[0]]
            tie_intro = (
                f'<div class="reco r-blue">'
                f'<h4>⚠️ Tied Detection: {names} ({tie_pct:.1f}% each)</h4>'
                f'<p>Multiple infestation stages share equal detection frequency. '
                f'Apply each recommendation below. Prioritise late-stage treatment '
                f'zones first, then address early-stage areas to prevent progression.</p>'
                f'</div>'
            )
            st.markdown(tie_intro, unsafe_allow_html=True)
            for k in leaders:
                st.markdown(_reco_card(k, pct[k]), unsafe_allow_html=True)
            # supplemental for others
            others = [k for k in active if k not in leaders]
            if others:
                supp = "**Also present (lower prevalence):** " + ", ".join(
                    f"{CLASS_ICON[k]} {CLASS_DISPLAY[k]} {pct[k]:.1f}%"
                    for k in others
                )
                st.markdown(supp)
            return

        # ── single dominant ──────────────────────────────────────────────
        dom    = leaders[0]
        others = [k for k in active if k != dom]

        # mixed-field note appended to dominant card
        mixed_note = ""
        if others:
            other_lines = "".join(
                f"<li>{CLASS_ICON[k]} <strong>{CLASS_DISPLAY[k]}</strong> "
                f"at {pct[k]:.1f}% — monitor for progression</li>"
                for k in sorted(others, key=lambda x: -pct[x])
            )
            mixed_note = (
                f'<p style="margin-top:10px;font-size:.82rem;'
                f'color:#5a7080"><strong>Also detected:</strong><ul>{other_lines}</ul></p>'
            )

        st.markdown(_reco_card(dom, pct[dom], extra=mixed_note), unsafe_allow_html=True)


# ─────────────────────────────────────────────────────────────────────────────
# 9.  EXPORT HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def build_exports():
    """Return (csv_bytes, json_bytes, summary_bytes) or (None,None,None)."""
    log = st.session_state.tracking_log
    if not log:
        return None, None, None

    df  = pd.DataFrame(log)
    csv = df.to_csv(index=False).encode("utf-8")

    counts = st.session_state.class_counts
    total  = sum(counts.values())
    summary = {
        "session_frames":    st.session_state.frame_count,
        "total_detections":  total,
        "class_counts":      counts,
        "class_percentages": {
            k: round(counts[k] / total * 100, 2) if total else 0
            for k in CLASS_KEYS
        },
        "detections": log,
    }
    jsn = json.dumps(summary, indent=2).encode("utf-8")
    return csv, jsn, summary


# ─────────────────────────────────────────────────────────────────────────────
# 10. SIDEBAR
# ─────────────────────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown("### ⚙️ Configuration")

    model_path  = st.text_input("Model path (.pt)", value="best.pt")
    conf_thresh = st.slider("Confidence threshold", 0.10, 0.95, 0.25, 0.05)
    iou_thresh  = st.slider("NMS IoU threshold",    0.10, 0.95, 0.45, 0.05)

    st.markdown("---")

    if st.button("🔄 Reset Session"):
        reset_session()
        st.rerun()

    st.markdown("---")
    st.markdown("**Class legend**")
    for k in CLASS_KEYS:
        c = CLASS_HEX[k]
        st.markdown(
            f'<div style="display:flex;align-items:center;gap:8px;margin:4px 0;">'
            f'<div style="width:13px;height:13px;border-radius:3px;background:{c}"></div>'
            f'<span style="font-size:.83rem">{CLASS_ICON[k]} {CLASS_DISPLAY[k]}</span>'
            f'</div>', unsafe_allow_html=True,
        )

    st.markdown("---")
    st.markdown("**💾 Export session data**")
    csv_b, jsn_b, summary = build_exports()
    if csv_b:
        st.download_button("📥 CSV report",  csv_b, "faw_detections.csv", "text/csv")
        st.download_button("📥 JSON report", jsn_b, "faw_detections.json","application/json")
    else:
        st.caption("Run a detection session first.")

# Load model (after sidebar so path widget is already rendered)
model, model_err = load_yolo(model_path)
if model_err:
    st.sidebar.warning(f"Model not loaded:\n{model_err}\n\nUsing mock detections.")
else:
    st.sidebar.success("✅ Model loaded")

# ─────────────────────────────────────────────────────────────────────────────
# 11. PAGE HEADER
# ─────────────────────────────────────────────────────────────────────────────
st.markdown("""
<div class="av-header">
  <h1>🌽 Agri-Vision · FAW Detection Dashboard</h1>
  <p>Real-time Fall Armyworm scouting · Image · Video · Webcam ·
     Aggregated recommendations · CSV / JSON export</p>
</div>
""", unsafe_allow_html=True)

# ─────────────────────────────────────────────────────────────────────────────
# 12. INPUT TABS
# ─────────────────────────────────────────────────────────────────────────────
tab_img, tab_vid, tab_cam = st.tabs(["📷 Image", "🎬 Video", "📹 Webcam"])

# ══════════════════════════════════════════════════════════════════════════════
# TAB — IMAGE
# ══════════════════════════════════════════════════════════════════════════════
with tab_img:
    uploaded = st.file_uploader(
        "Upload crop / leaf image",
        type=["jpg","jpeg","png","bmp","tiff","webp"],
        key="img_up",
    )
    if uploaded:
        col_l, col_r = st.columns(2, gap="medium")
        pil_orig = Image.open(uploaded).convert("RGB")

        with col_l:
            st.markdown("**Original**")
            st.image(pil_orig, use_container_width=True)

        run_img = st.button("▶️ Run Detection", key="btn_img")
        if run_img:
            # reset counts so image session is isolated
            reset_session()
            st.session_state.session_start = time.strftime("%Y-%m-%d %H:%M:%S")

            frame_rgb = np.array(pil_orig)
            frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

            with st.spinner("Analysing image…"):
                dets = run_inference(frame_rgb, model, conf_thresh, iou_thresh)

            annotated_bgr = draw_boxes(frame_bgr, dets)
            annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)

            with col_r:
                st.markdown("**Detections**")
                st.image(annotated_rgb, use_container_width=True)

                # Download annotated
                buf = io.BytesIO()
                Image.fromarray(annotated_rgb).save(buf, format="JPEG", quality=93)
                st.download_button(
                    "⬇️ Download annotated image",
                    buf.getvalue(), "faw_annotated.jpg", "image/jpeg",
                )

            accumulate(dets, 1, "image")

            # Detection table
            if dets:
                st.markdown("---")
                import pandas as _pd
                df_d = _pd.DataFrame([{
                    "#":           i+1,
                    "Class":       f"{CLASS_ICON[d['key']]} {CLASS_DISPLAY[d['key']]}",
                    "Confidence":  f"{d['conf']:.1%}",
                    "x1":d["box"][0],"y1":d["box"][1],
                    "x2":d["box"][2],"y2":d["box"][3],
                } for i,d in enumerate(dets)])
                st.dataframe(df_d, use_container_width=True, hide_index=True)

        # Analytics panel (shown after any run)
        st.markdown("---")
        analytics_panel_img = st.empty()
        render_analytics(
            analytics_panel_img,
            st.session_state.class_counts,
            sum(st.session_state.class_counts.values()),
            st.session_state.frame_count,
        )

# ══════════════════════════════════════════════════════════════════════════════
# TAB — VIDEO
# ══════════════════════════════════════════════════════════════════════════════
with tab_vid:
    vid_file = st.file_uploader(
        "Upload drone / field video",
        type=["mp4","avi","mov","mkv","webm"],
        key="vid_up",
    )

    skip_n = st.number_input(
        "Process every N-th frame (1 = all, higher = faster)",
        min_value=1, max_value=15, value=2,
    )

    if vid_file:
        run_vid = st.button("▶️ Process Video", key="btn_vid")
        if run_vid:
            reset_session()
            st.session_state.session_start = time.strftime("%Y-%m-%d %H:%M:%S")

            # Write to safe temp file
            tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".mp4")
            tmp.write(vid_file.read()); tmp.flush(); tmp.close()

            cap = cv2.VideoCapture(tmp.name)
            if not cap.isOpened():
                st.error("Could not open video file.")
                os.unlink(tmp.name)
            else:
                total_fr  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
                fps_v     = cap.get(cv2.CAP_PROP_FPS) or 25
                w_v       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                h_v       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
                st.info(f"Video: {w_v}×{h_v}px · {fps_v:.0f} fps · {total_fr} frames")

                vid_col, ana_col = st.columns([3, 2], gap="medium")
                with vid_col:
                    vid_ph = st.empty()
                with ana_col:
                    ana_ph = st.empty()

                prog = st.progress(0, text="Starting…")
                fidx = 0

                try:
                    while True:
                        ret, frame_bgr = cap.read()
                        if not ret:
                            break

                        fidx += 1
                        prog.progress(
                            min(fidx / total_fr, 1.0),
                            text=f"Frame {fidx}/{total_fr}",
                        )

                        if fidx % skip_n != 0:
                            continue

                        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
                        dets      = run_inference(frame_rgb, model, conf_thresh, iou_thresh)
                        annotated = draw_boxes(frame_bgr, dets)

                        vid_ph.image(
                            cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB),
                            use_container_width=True,
                            caption=f"Frame {fidx}",
                        )
                        accumulate(dets, fidx, "video")
                        render_analytics(
                            ana_ph,
                            st.session_state.class_counts,
                            sum(st.session_state.class_counts.values()),
                            st.session_state.frame_count,
                        )

                finally:
                    cap.release()
                    os.unlink(tmp.name)

                prog.empty()
                st.success(
                    f"✅ Done — {st.session_state.frame_count} frames analysed · "
                    f"{sum(st.session_state.class_counts.values())} total detections"
                )

        # Post-processing summary (persists after loop ends)
        if st.session_state.frame_count > 0 and vid_file:
            st.markdown("---")
            st.markdown("#### 📊 Full Session Summary")
            summary_ph_v = st.empty()
            render_analytics(
                summary_ph_v,
                st.session_state.class_counts,
                sum(st.session_state.class_counts.values()),
                st.session_state.frame_count,
            )

# ══════════════════════════════════════════════════════════════════════════════
# TAB — WEBCAM
# ══════════════════════════════════════════════════════════════════════════════
with tab_cam:
    st.markdown(
        "Each captured frame is analysed. Detections accumulate across the "
        "session. Press **Stop & Summarise** to finalise recommendations."
    )

    cam_col, ana_col = st.columns([3, 2], gap="medium")

    with cam_col:
        # Streamlit's camera_input: one click = one frame, works on all devices
        cam_frame = st.camera_input("📸 Capture frame", key="cam_input")

        c1, c2 = st.columns(2)
        with c1:
            stop_cam = st.button("🛑 Stop & Summarise", key="btn_stop_cam")
        with c2:
            if st.button("🔄 Reset camera session", key="btn_reset_cam"):
                reset_session()
                st.rerun()

    with ana_col:
        cam_ana_ph = st.empty()

    if cam_frame:
        if st.session_state.session_start is None:
            st.session_state.session_start = time.strftime("%Y-%m-%d %H:%M:%S")

        pil_cam   = Image.open(cam_frame).convert("RGB")
        frame_rgb = np.array(pil_cam)
        frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

        with st.spinner("Analysing frame…"):
            dets = run_inference(frame_rgb, model, conf_thresh, iou_thresh)

        annotated_rgb = cv2.cvtColor(draw_boxes(frame_bgr, dets), cv2.COLOR_BGR2RGB)

        with cam_col:
            st.image(annotated_rgb, caption="Annotated frame", use_container_width=True)
            frame_counts = collections.Counter(d["key"] for d in dets)
            if frame_counts:
                fc_str = " · ".join(
                    f"{CLASS_ICON[k]} {CLASS_DISPLAY[k]} ×{n}"
                    for k, n in sorted(frame_counts.items())
                )
                st.caption(f"This frame: {fc_str}")
            else:
                st.caption("This frame: no detections")

        accumulate(dets, st.session_state.frame_count + 1, "webcam")

    # Always render live analytics pane
    render_analytics(
        cam_ana_ph,
        st.session_state.class_counts,
        sum(st.session_state.class_counts.values()),
        st.session_state.frame_count,
    )

    # Final summary section after stop
    if stop_cam and st.session_state.frame_count > 0:
        st.markdown("---")
        st.markdown("#### 📊 Session Summary")
        summary_ph_c = st.empty()
        render_analytics(
            summary_ph_c,
            st.session_state.class_counts,
            sum(st.session_state.class_counts.values()),
            st.session_state.frame_count,
        )
        csv_b, jsn_b, _ = build_exports()
        if csv_b:
            dl1, dl2 = st.columns(2)
            with dl1:
                st.download_button(
                    "📥 Download CSV",  csv_b, "faw_webcam_session.csv",  "text/csv"
                )
            with dl2:
                st.download_button(
                    "📥 Download JSON", jsn_b, "faw_webcam_session.json", "application/json"
                )


!streamlit run app.py &>/dev/null &
!npx localtunnel --port 8501

2026-06-08 21:07:10.768 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-08 21:07:10.770 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-08 21:07:10.773 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-08 21:07:10.774 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-08 21:07:10.777 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-08 21:07:10.779 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-08 21:07:10.780 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-08 21:07:10.782 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


ERROR:pyngrok.process.ngrok:t=2026-06-08T21:07:13+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: YOUR_NGROK_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
ERROR:pyngrok.process.ngrok:t=2026-06-08T21:07:13+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: YOUR_NGROK_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
ERROR:pyngrok.process.ngrok:t=2026-06-08T21:07:13+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nY

PyngrokNgrokError: The ngrok process errored on start: authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: YOUR_NGROK_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n.



2026-06-08 20:44:05.596 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.59.101.204:8501


  Stopping...
